

### Enable CDC

```
EXEC sys.sp_cdc_enable_db;
SELECT is_cdc_enabled FROM sys.databases WHERE name = DB_NAME();
```



Now create a sample schema and table. insert records..

using  gks as schema name for sql server, if account shared, use differnt initial

```
CREATE SCHEMA  gks;

CREATE TABLE gks.products (
    id INT PRIMARY KEY,
    name NVARCHAR(100),
    price DECIMAL(10, 2),
    stock INT
);
```




enable cdc on table products

```
EXEC sys.sp_cdc_enable_table
    @source_schema = 'gks',
    @source_name = 'products',
    @role_name = NULL;

```

verify it is enabled..
```
SELECT * FROM cdc.change_tables;
```



insert few records

```
INSERT INTO gks.products (id, name, price, stock) VALUES (1, 'Laptop', 999.99, 10);
INSERT INTO gks.products (id, name, price, stock) VALUES (2, 'Mouse', 19.99, 50);
INSERT INTO gks.products (id, name, price, stock) VALUES (3, 'Keyboard', 49.99, 30);
```

-- _$operation, 1 means delete, 2 means insert, 3 means before update, 4 means after 

check whether your changes are captured

```
SELECT *
FROM cdc.gks_products_CT
ORDER BY __$start_lsn;
```

-- Update price of Laptop

```
UPDATE gks.products SET price = 899.99 WHERE id = 1;
```

check changes in cdc

```
SELECT *
FROM cdc.gks_products_CT
ORDER BY __$start_lsn;
```

-- Add a new product

```
INSERT INTO gks.products (id, name, price, stock) VALUES (4, 'Monitor', 199.99, 20);
```

check changes in cdc

```
SELECT *
FROM cdc.gks_products_CT
ORDER BY __$start_lsn;
```

-- Delete a product
```
DELETE FROM gks.products WHERE id = 2;
```